# 逐手模型微調與推論

`aicup/train_permove.py` 微調 KataGo humanv0（10 份 policy head，每個等級一份），`aicup/infer_permove.py` 輸出每局 `pm`（目標側）與 `pmo`（對手側）。

- 這份 notebook 是實驗，不在競賽一最終版本流程內。
- 精度：`train_permove --amp auto`，A100 用 bf16，T4 用 float32；推論固定 float32
- 外部資源：KataGo `python/`（tag `v1.15.0`，MIT）與 `b18c384nbt-humanv0.ckpt`，由程式從官方網址下載，快取在 Drive 的 `katago/`
- 特徵先複製到 `/content`，訓練與推論期間不讀 Drive 上的特徵
- checkpoint 直接寫在 Drive 的 `rank-trained-models/permove/<名稱>/`。斷線時重開 runtime、依序執行全部 cell，會從 `last.pt` 接續

| MODE | 內容 |
|---|---|
| `smoke` | 2,000 個局面訓練 30 步，再接續到 35 步，最後比較推論的 float32 與 bf16。重跑前先刪掉 `permove/smoke/` |
| `train` | spike 資料 1 個 epoch，checkpoint 在 `permove/train/` |
| `infer_val` | val648 兩側推論，輸出 `humansl-analysis/val648-pm.jsonl` |
| `infer_test` | 測試集兩側推論，輸出 `humansl-analysis/test-pm.jsonl` |

- `BATCH`：訓練的 batch。T4 放不下 256 時改成 128，lr 不變。

## 需要的檔案

路徑都在 Drive 的 `MyDrive/aicup-2026/` 下，在本機產生後上傳。

| 路徑 | MODE | 產生方式（本機，repo 根目錄） |
|---|---|---|
| `permove-features/spike/` | `smoke`、`train` | `python -m aicup.extract_permove spike --out permove-features/spike` |
| `permove-features/val648/` | `smoke`、`infer_val` | `python -m aicup.extract_permove val648 --out permove-features/val648` |
| `permove-features/test/` | `infer_test` | `python -m aicup.extract_permove val648 --out permove-features/test --test-dir tests --keys-from humansl-analysis/test-p3.jsonl --opp-from humansl-analysis/test-p3-opp.jsonl --games-per-shard 100` |
| `humansl-analysis/val648-p3.jsonl` | `smoke`、`infer_val` | `colab_humansl.ipynb` 的 `val648` |
| `humansl-analysis/test-p3.jsonl` | `infer_test` | `colab_humansl.ipynb` 的 `test` |

## 參數

In [ ]:
REPO_URL = 'https://github.com/Jx-study/aicup-2026-go.git'
COMMIT = 'origin/main'  # 重現特定版本時填 commit hash
MODE = 'smoke'
BATCH = 256
AUTO_UNASSIGN = True

PM_HINT = 'python -m aicup.extract_permove %s --out permove-features/%s'
HSL_HINT = 'colab_humansl.ipynb 的 MODE %s'
NEEDS = {  # MODE: [(Drive 上的路徑, 產生方式)]
    'smoke': [('permove-features/spike', PM_HINT % ('spike', 'spike')), ('permove-features/val648', PM_HINT % ('val648', 'val648')),
              ('humansl-analysis/val648-p3.jsonl', HSL_HINT % 'val648')],
    'train': [('permove-features/spike', PM_HINT % ('spike', 'spike'))],
    'infer_val': [('permove-features/val648', PM_HINT % ('val648', 'val648')),
                  ('humansl-analysis/val648-p3.jsonl', HSL_HINT % 'val648')],
    'infer_test': [('permove-features/test', 'python -m aicup.extract_permove val648 --out permove-features/test --test-dir tests '
                    '--keys-from humansl-analysis/test-p3.jsonl --opp-from humansl-analysis/test-p3-opp.jsonl '
                    '--games-per-shard 100'),
                   ('humansl-analysis/test-p3.jsonl', HSL_HINT % 'test')],
}
if MODE not in NEEDS:
    raise SystemExit('MODE 要是 %s 之一' % ', '.join(NEEDS))
NEEDS = NEEDS[MODE]
CKPT_ROOT = 'rank-trained-models/permove'
HUMANV0 = '/content/models/b18c384nbt-humanv0.ckpt'
TRAIN = ['python', '-m', 'aicup.train_permove', '--data', '/content/permove-features/spike', '--humanv0', HUMANV0,
         '--batch', str(BATCH)]
INFER = {'infer_val': ('/content/permove-features/val648/val', 'val648-pm.jsonl', 'val648-p3.jsonl'),
         'infer_test': ('/content/permove-features/test/val', 'test-pm.jsonl', 'test-p3.jsonl')}
if MODE == 'smoke':
    CKPT = CKPT_ROOT + '/smoke'
    SMOKE = TRAIN + ['--ckpt-dir', CKPT, '--limit', '2000']
    CMDS = [(SMOKE + ['--max-steps', '30', '--log-every', '10'], CKPT + '/train.log'),
            (SMOKE + ['--max-steps', '35', '--log-every', '5'], CKPT + '/train.log'),
            (['python', '-m', 'aicup.infer_permove', '--model', CKPT + '/model.pt', '--data',
              '/content/permove-features/val648/val', '--compare-dtypes', '20'], None)]
elif MODE == 'train':
    CKPT = CKPT_ROOT + '/train'
    CMDS = [(TRAIN + ['--ckpt-dir', CKPT, '--log-every', '200'], CKPT + '/train.log')]
else:
    DATA, OUT_NAME, REF = INFER[MODE]
    CMDS = [(['python', '-m', 'aicup.infer_permove', '--model', CKPT_ROOT + '/train/model.pt', '--data', DATA,
              '--out', '/content/' + OUT_NAME], '/content/%s.log' % MODE)]
for cmd, _ in CMDS:
    print(' '.join(cmd))

## 取得程式

要看到：Drive 掛載完成、最新的 commit、GPU 型號。

In [ ]:
import os, subprocess, sys
from google.colab import drive

drive.mount('/content/drive')
REPO = '/content/aicup-2026-go'
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '-q', REPO_URL, REPO], check=True)
os.chdir(REPO)
subprocess.run(['git', 'fetch', '-q', 'origin'], check=True)
subprocess.run(['git', 'checkout', '-q', COMMIT], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sgfmill'], check=True)
from aicup.colab_nb import DRIVE_ROOT, check_commit, link_drive, need, run

GIT_HEAD = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip()

!git log --oneline -1
!nvidia-smi -L

## 準備資料

要看到：需要的檔案都在、複製的清單、每個目錄的分片數、humanv0 的路徑。

In [ ]:
link_drive(['rank-trained-models'])
import shutil

need([(os.path.join(DRIVE_ROOT, rel), how) for rel, how in NEEDS])
for rel, _ in NEEDS:
    src, dst = os.path.join(DRIVE_ROOT, rel), '/content/' + rel
    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy(src, dst)
    print('複製', rel)
!find /content/permove-features -name '*.npz' | sed 's|/[^/]*$||' | sort | uniq -c
KATAGO = '/content/KataGo'
KATAGO_COMMIT = 'c560d38585b446bc93b0a62c77364bc8e7d266a4'  # tag v1.15.0
if not os.path.exists(KATAGO + '/python/features.py'):
    subprocess.run(['git', 'clone', '-q', '--depth', '1', '--branch', 'v1.15.0', '--filter=blob:none', '--sparse',
                    'https://github.com/lightvector/KataGo', KATAGO], check=True)
    subprocess.run(['git', '-C', KATAGO, 'sparse-checkout', 'set', 'python'], check=True)
head = subprocess.run(['git', '-C', KATAGO, 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
assert head == KATAGO_COMMIT, 'KataGo 的 commit 不是 v1.15.0'
os.environ['KATAGO_PYTHON'] = KATAGO + '/python'

# katagotraining.org 會擋 urllib，先用 Drive 快取，沒有才用 wget
HUMANV0 = '/content/models/b18c384nbt-humanv0.ckpt'
CACHE = DRIVE_ROOT + '/katago/b18c384nbt-humanv0.ckpt'
os.makedirs(os.path.dirname(HUMANV0), exist_ok=True)
if not os.path.exists(HUMANV0):
    if os.path.exists(CACHE):
        shutil.copy(CACHE, HUMANV0)
    else:
        !wget -q -O {HUMANV0} https://media.katagotraining.org/uploaded/networks/models_extra/b18c384nbt-humanv0.ckpt
from aicup import katago_torch as kt

kt.ensure_humanv0(HUMANV0)  # 檢查大小與 SHA-256
if not os.path.exists(CACHE):
    os.makedirs(os.path.dirname(CACHE), exist_ok=True)
    shutil.copy(HUMANV0, CACHE)
print('humanv0', HUMANV0)

## 執行

- `smoke`：看 loss 是否下降、第二次執行是否印出「從 ... 接續，第 30 步」、每秒局面數，以及 float32 與 bf16 的每局特徵差最大值。
- `train`：log 同時寫到 checkpoint 目錄的 `train.log`。中途斷線時重跑本 cell 會接續。
- `infer_*`：推論只在最後寫檔，斷線時要整個重跑。

In [ ]:
for cmd, log in CMDS:
    run(cmd, log)
if MODE.startswith('infer_'):
    os.makedirs(DRIVE_ROOT + '/humansl-analysis', exist_ok=True)
    for name in (OUT_NAME, MODE + '.log'):
        shutil.copy('/content/' + name, DRIVE_ROOT + '/humansl-analysis/')

## 檢查

- `smoke`、`train`：列出 holdout 的 loss。前 30% 的步數內 policy loss 比第 0 步高超過 5% 時，停止並回報。原因：主幹的 lr 可能太大。
- `infer_*`：行數與 key 集合要和參考 jsonl 相同，每行都要有 `pm`、`pmo`。

In [ ]:
import json

if MODE in ('smoke', 'train'):
    hist = [json.loads(line) for line in open(CKPT + '/history.jsonl')]
    for h in hist:
        print('step %6d  policy %.4f  rank %.4f  acc %.3f' % (h['step'], h['policy_loss'], h['rank_loss'], h['rank_acc']))
    base, last = hist[0]['policy_loss'], hist[-1]['step']
    early = [h for h in hist if 0 < h['step'] <= 0.3 * last]
    if any(h['policy_loss'] > 1.05 * base for h in early):
        print('停止：前 30% 的步數內 holdout policy loss 比第 0 步高超過 5%')
else:
    want = {json.loads(line)['key'] for line in open('/content/humansl-analysis/' + REF)}
    rows = [json.loads(line) for line in open(DRIVE_ROOT + '/humansl-analysis/' + OUT_NAME)]
    got = {r['key'] for r in rows}
    bad = [r['key'] for r in rows if r['pm'] is None or r['pmo'] is None]
    print('%d 行，key 和 %s 相同：%s，缺 pm 或 pmo：%d 局' % (len(rows), REF, got == want and len(rows) == len(want), len(bad)))

## 釋放 runtime

跑完才執行。程式結束後 runtime 不會自動中斷，閒置也會扣運算單元。

In [ ]:
from google.colab import runtime

drive.flush_and_unmount()  # 等 Drive 寫入完成
if AUTO_UNASSIGN:
    runtime.unassign()
else:
    print('未釋放 runtime，用完記得手動中斷')